# 05. Named Entity Recognition with BERT Using Token Classification


## 1. What you will build
- A full NER workflow with DistilBERT token classification (`PER`, `ORG`, `LOC`, `DATE`, `MONEY`).
- BIO-tag creation from external CSV files in `data/05_data`.
- Reproducible training and inference with business-like text.
- A side-by-side evaluation table comparing `spaCy` (zero-shot) vs fine-tuned `BERT` on the same test split.


## 2. When to use this in real companies
Use this approach when generic NER is not enough and teams need extraction adapted to internal language patterns (incident logs, finance notes, compliance updates).

Typical enterprise situations:
- You must extract fields from semi-structured operational text with consistent KPIs.
- You need measurable quality gains over off-the-shelf NER.
- You want one notebook that shows both baseline (`spaCy`) and task-specific (`BERT`) performance.


## 3. Business goal
Train and evaluate a BERT-based NER model for `PER`, `ORG`, `LOC`, `DATE`, and `MONEY`, then compare it against `spaCy en_core_web_sm` using exact entity-level metrics.


<h2 style="color:#f8fafc; font-family:Arial, sans-serif; margin-bottom: 16px;">
  NER with spaCy vs NER with Fine-Tuned BERT
</h2>

<table style="width:100%; border-collapse:collapse; font-family:Arial, sans-serif; background:#111827; color:#e5e7eb;">

  <tr style="background:#2563eb; color:white;">
    <th style="padding:12px; border:1px solid #374151; text-align:left;">Aspect</th>
    <th style="padding:12px; border:1px solid #374151; text-align:left;">NER with spaCy</th>
    <th style="padding:12px; border:1px solid #374151; text-align:left;">NER with Fine-Tuned BERT</th>
  </tr>

  <tr>
    <td style="padding:12px; border:1px solid #374151;"><strong>How it starts</strong></td>
    <td style="padding:12px; border:1px solid #374151;">Uses a pre-trained NER pipeline (e.g., <code>en_core_web_sm</code>).</td>
    <td style="padding:12px; border:1px solid #374151;">Starts from a pre-trained language model (e.g., <code>distilbert-base-cased</code>) plus a new token-classification head.</td>
  </tr>

  <tr style="background:#1f2937;">
    <td style="padding:12px; border:1px solid #374151;"><strong>How it learns NER labels</strong></td>
    <td style="padding:12px; border:1px solid #374151;">Already learned generic labels in pretraining package.</td>
    <td style="padding:12px; border:1px solid #374151;">Learns your labels during fine-tuning from BIO tags (<code>B-PER</code>, <code>I-ORG</code>, etc.) and <code>label2id/id2label</code>.</td>
  </tr>

  <tr>
    <td style="padding:12px; border:1px solid #374151;"><strong>Need labeled data?</strong></td>
    <td style="padding:12px; border:1px solid #374151;">Not required for zero-shot usage.</td>
    <td style="padding:12px; border:1px solid #374151;">Yes, required for good task/domain performance.</td>
  </tr>

  <tr style="background:#1f2937;">
    <td style="padding:12px; border:1px solid #374151;"><strong>Do you need all names in the world?</strong></td>
    <td style="padding:12px; border:1px solid #374151;">No.</td>
    <td style="padding:12px; border:1px solid #374151;">No. You need representative domain examples, not every possible name.</td>
  </tr>

  <tr>
    <td style="padding:12px; border:1px solid #374151;"><strong>Context understanding</strong></td>
    <td style="padding:12px; border:1px solid #374151;">Good for common/general patterns.</td>
    <td style="padding:12px; border:1px solid #374151;">Stronger contextual understanding and better adaptation to internal language.</td>
  </tr>

  <tr style="background:#1f2937;">
    <td style="padding:12px; border:1px solid #374151;"><strong>Inference speed</strong></td>
    <td style="padding:12px; border:1px solid #374151;">Fast (CPU-friendly).</td>
    <td style="padding:12px; border:1px solid #374151;">Slower than spaCy (benefits from GPU).</td>
  </tr>

  <tr>
    <td style="padding:12px; border:1px solid #374151;"><strong>Computational cost</strong></td>
    <td style="padding:12px; border:1px solid #374151;">Low.</td>
    <td style="padding:12px; border:1px solid #374151;">Higher training/inference cost.</td>
  </tr>

  <tr style="background:#1f2937;">
    <td style="padding:12px; border:1px solid #374151;"><strong>Best use case</strong></td>
    <td style="padding:12px; border:1px solid #374151;">Quick baseline and production with strict latency/cost limits.</td>
    <td style="padding:12px; border:1px solid #374151;">Business-critical extraction quality and domain-specific entity language.</td>
  </tr>

  <tr>
    <td style="padding:12px; border:1px solid #374151;"><strong>Output</strong></td>
    <td style="padding:12px; border:1px solid #374151;">Entity spans + labels from spaCy pipeline.</td>
    <td style="padding:12px; border:1px solid #374151;">Token BIO predictions aggregated to entity spans.</td>
  </tr>

</table>

<p style="margin-top:16px; color:#9ca3af; font-family:Arial;">
  In short: spaCy is ideal for speed and simplicity; fine-tuned BERT is ideal when you need higher domain-specific NER quality and can afford labeled data + training cost.
</p>

## 4. Imports and setup


In [1]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import evaluate
import spacy

from datasets import Dataset
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments,
    pipeline,
)
from IPython.display import HTML, display

# Keep notebook output clean for teaching sessions.
warnings.filterwarnings("ignore", category=UserWarning, module="torch.utils.data.dataloader")

## 5. Load datasets from `data/05_data`


In [2]:
DOCS_PATH = Path("../data/05_data/ner_bert_documents.csv")
GOLD_PATH = Path("../data/05_data/ner_bert_gold_entities.csv")

docs_df = pd.read_csv(DOCS_PATH)
gold_df = pd.read_csv(GOLD_PATH)

print(f"Documents: {len(docs_df):,}")
print(f"Gold entities: {len(gold_df):,}")
docs_df.head()

Documents: 600
Gold entities: 3,000


,doc_id,text
0,NER5-0001,Status update INC-2679: Polar Manufacturing co...
1,NER5-0002,"Control note: in Berlin, Isaac Chen from Apex ..."
2,NER5-0003,Northwind Group confirmed a deployment update ...
3,NER5-0004,Daniel Nguyen led a cost optimization checkpoi...
4,NER5-0005,During the integration review in Stockholm on ...


## 6. Build BIO tags from document-level entity annotations


In [3]:
nlp_tokenizer = spacy.blank("en")
base_labels = ["PER", "ORG", "LOC", "DATE", "MONEY"]


def build_bio_for_doc(text: str, entities: list[tuple[str, str]]):
    """Convert one document with entity strings into BIO token labels."""
    doc = nlp_tokenizer(text)
    tokens = [t.text for t in doc]
    tags = ["O"] * len(doc)
    occupied = set()

    # Match longer entities first and avoid overlapping spans.
    sorted_entities = sorted(entities, key=lambda x: len(str(x[0])), reverse=True)
    for entity_text, entity_label in sorted_entities:
        if not isinstance(entity_text, str) or not entity_text.strip():
            continue

        for match in re.finditer(re.escape(entity_text), text):
            start, end = match.span()
            token_ids = [
                i for i, tok in enumerate(doc)
                if (tok.idx < end) and (tok.idx + len(tok.text) > start)
            ]
            if not token_ids or any(i in occupied for i in token_ids):
                continue

            tags[token_ids[0]] = f"B-{entity_label}"
            for idx in token_ids[1:]:
                tags[idx] = f"I-{entity_label}"
            occupied.update(token_ids)
            break

    return tokens, tags


records = []
for _, row in docs_df.iterrows():
    doc_id = row["doc_id"]
    text = row["text"]
    entities = gold_df.loc[gold_df["doc_id"] == doc_id, ["entity_text", "label"]].values.tolist()
    tokens, tags = build_bio_for_doc(text, entities)
    records.append({"doc_id": doc_id, "tokens": tokens, "ner_tags": tags})

bio_df = pd.DataFrame(records)
bio_df.head(3)

,doc_id,tokens,ner_tags
0,NER5-0001,"[Status, update, INC-2679, :, Polar, Manufactu...","[O, O, O, O, B-ORG, I-ORG, O, O, B-PER, I-PER,..."
1,NER5-0002,"[Control, note, :, in, Berlin, ,, Isaac, Chen,...","[O, O, O, O, B-LOC, O, B-PER, I-PER, O, B-ORG,..."
2,NER5-0003,"[Northwind, Group, confirmed, a, deployment, u...","[B-ORG, I-ORG, O, O, O, O, O, B-PER, I-PER, O,..."


In [4]:
all_tags = sorted({tag for tags in bio_df["ner_tags"] for tag in tags})
tag2id = {tag: i for i, tag in enumerate(all_tags)}
id2tag = {i: tag for tag, i in tag2id.items()}

bio_df["ner_tag_ids"] = bio_df["ner_tags"].apply(lambda tags: [tag2id[t] for t in tags])

print(f"Tag set ({len(all_tags)}):", all_tags)

Tag set (11): ['B-DATE', 'B-LOC', 'B-MONEY', 'B-ORG', 'B-PER', 'I-DATE', 'I-LOC', 'I-MONEY', 'I-ORG', 'I-PER', 'O']


## 7. Train/validation/test split


In [5]:
train_df, temp_df = train_test_split(bio_df, test_size=0.3, random_state=42)
valid_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

train_df = train_df.reset_index(drop=True)
valid_df = valid_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"Train: {len(train_df)} | Valid: {len(valid_df)} | Test: {len(test_df)}")

Train: 420 | Valid: 90 | Test: 90


In [6]:
def to_dataset(frame: pd.DataFrame) -> Dataset:
    return Dataset.from_dict(
        {
            "doc_id": frame["doc_id"].tolist(),
            "tokens": frame["tokens"].tolist(),
            "ner_tags": frame["ner_tag_ids"].tolist(),
        }
    )

train_ds = to_dataset(train_df)
valid_ds = to_dataset(valid_df)
test_ds = to_dataset(test_df)

train_ds

Dataset({
    features: ['doc_id', 'tokens', 'ner_tags'],
    num_rows: 420
})

## 8. Tokenize and align labels for BERT


In [7]:
model_ckpt = "distilbert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)


def tokenize_and_align(examples):
    tokenized = tokenizer(
        examples["tokens"],
        truncation=True,
        max_length=128,
        is_split_into_words=True,
    )
    aligned_labels = []

    for i, labels in enumerate(examples["ner_tags"]):
        word_ids = tokenized.word_ids(batch_index=i)
        prev_word_id = None
        label_ids = []
        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)
            elif word_id != prev_word_id:
                label_ids.append(labels[word_id])
            else:
                label_ids.append(-100)
            prev_word_id = word_id

        aligned_labels.append(label_ids)

    tokenized["labels"] = aligned_labels
    return tokenized


train_tok = train_ds.map(tokenize_and_align, batched=True)
valid_tok = valid_ds.map(tokenize_and_align, batched=True)
test_tok = test_ds.map(tokenize_and_align, batched=True)

Map:   0%|          | 0/420 [00:00<?, ? examples/s]

Map:   0%|          | 0/90 [00:00<?, ? examples/s]

Map:   0%|          | 0/90 [00:00<?, ? examples/s]

## 9. Fine-tune BERT token classifier

Note: when loading from `distilbert-base-cased`, messages about missing token-classification head weights are expected.
Those task-specific layers are initialized and then trained on your NER data, which is the correct behavior.


In [8]:
model = AutoModelForTokenClassification.from_pretrained(
    model_ckpt,
    num_labels=len(all_tags),
    id2label=id2tag,
    label2id=tag2id,
)

collator = DataCollatorForTokenClassification(tokenizer=tokenizer)
metric = evaluate.load("seqeval")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=2)

    true_preds, true_labels = [], []
    for pred_seq, label_seq in zip(preds, labels):
        p_tags, l_tags = [], []
        for p, l in zip(pred_seq, label_seq):
            if l != -100:
                p_tags.append(id2tag[p])
                l_tags.append(id2tag[l])
        true_preds.append(p_tags)
        true_labels.append(l_tags)

    out = metric.compute(predictions=true_preds, references=true_labels)
    return {
        "precision": out["overall_precision"],
        "recall": out["overall_recall"],
        "f1": out["overall_f1"],
        "accuracy": out["overall_accuracy"],
    }


args = TrainingArguments(
    output_dir="artifacts/05_bert_ner",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="no",
    logging_strategy="epoch",
    report_to="none",
    dataloader_pin_memory=False,
    disable_tqdm=False,
    seed=42,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=valid_tok,
    processing_class=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

trainer.train()
test_pred_output = trainer.predict(test_tok)
ner_eval = test_pred_output.metrics
ner_eval

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-cased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,1.035312,0.143221,0.862500,0.920000,0.890323,0.980443
2,0.059096,0.017771,1.000000,1.000000,1.000000,1.000000


{'test_loss': 0.016772529110312462,
 'test_precision': 1.0,
 'test_recall': 1.0,
 'test_f1': 1.0,
 'test_accuracy': 1.0,
 'test_runtime': 8.3269,
 'test_samples_per_second': 10.808,
 'test_steps_per_second': 1.441}

## 10. Inference example


In [9]:
ner_pipe = pipeline(
    "token-classification",
    model=trainer.model,
    tokenizer=tokenizer,
    aggregation_strategy="simple",
)

sample_text = "Control note: in Madrid, Emma Rossi from Atlas Security logged a deployment exception on 2026-07-21, linked to USD 18,900."
pd.DataFrame(ner_pipe(sample_text))[["word", "entity_group", "score"]]

,word,entity_group,score
0,Madrid,LOC,0.983265
1,Emma Rossi,PER,0.980005
2,Atlas Security,ORG,0.960122
3,2026 - 07 - 21,DATE,0.985836
4,"USD 18, 900",MONEY,0.907561


## 11. Summary
- `data/05_data` contains a larger synthetic corpus with paraphrased enterprise text.
- BERT is trained with stronger defaults (`num_train_epochs=2`, `learning_rate=2e-5`, `max_length=128`, warmup and weight decay).
- The notebook now reports both BERT token-level metrics and an entity-level comparison against spaCy on the same test split.
